In [ ]:
!pip install tqdm


In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.applications import ResNet152V2
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Concatenate, Dropout
from tensorflow.keras.models import Model
from tensorflow.data import AUTOTUNE
from tqdm import tqdm
import matplotlib.pyplot as plt

# === Configuration ===
img_size = (224, 224)
train_dir = 'data/train'
test_dir = 'data/test'
train_csv = 'output_csv/test_biomarkers.csv'
test_csv = 'output_csv/test_biomarkers.csv'
batch_size = 16
epochs = 10

# === Load Biomarkers from CSV ===
def load_biomarkers(train_csv, test_csv):
    train_df = pd.read_csv(train_csv)
    test_df = pd.read_csv(test_csv)

    train_df = train_df.select_dtypes(include=[np.number])
    test_df = test_df.select_dtypes(include=[np.number])

    assert train_df.shape[1] == 137, "Train CSV must have 136 features + label"
    assert test_df.shape[1] == 137, "Test CSV must have 136 features + label"

    X_train_bio = train_df.iloc[:, :-1].values.astype(np.float32)
    y_train = train_df.iloc[:, -1].values.astype(np.float32)

    X_test_bio = test_df.iloc[:, :-1].values.astype(np.float32)
    y_test = test_df.iloc[:, -1].values.astype(np.float32)

    return X_train_bio, X_test_bio, y_train, y_test

# === LBP Feature Extractor ===
def extract_lbp_features(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    lbp = cv2.calcHist([gray], [0], None, [256], [0, 256])
    lbp = lbp.flatten() / np.sum(lbp)
    return lbp.astype(np.float32)

# === Image + LBP Generator ===
class HybridImageLBPGenerator(tf.keras.utils.Sequence):
    def __init__(self, directory, biomarker_data, labels, batch_size, img_size):
        self.filepaths = []
        self.labels = labels
        self.biomarker_data = biomarker_data
        self.batch_size = batch_size
        self.img_size = img_size
        class_dirs = ['fake', 'real']
        for label_idx, class_name in enumerate(class_dirs):
            class_path = os.path.join(directory, class_name)
            files = sorted(os.listdir(class_path))
            self.filepaths += [os.path.join(class_path, f) for f in files if f.endswith(('.jpg', '.png'))]

    def __len__(self):
        return len(self.filepaths) // self.batch_size

    def __getitem__(self, idx):
        batch_paths = self.filepaths[idx * self.batch_size: (idx + 1) * self.batch_size]
        batch_imgs = []
        batch_lbps = []
        batch_bio = self.biomarker_data[idx * self.batch_size: (idx + 1) * self.batch_size]
        batch_labels = self.labels[idx * self.batch_size: (idx + 1) * self.batch_size]

        for path in batch_paths:
            image = cv2.imread(path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, self.img_size)
            lbp = extract_lbp_features(image)
            image = image / 255.0
            batch_imgs.append(image)
            batch_lbps.append(lbp)

        return [np.array(batch_imgs), np.array(batch_bio), np.array(batch_lbps)], np.array(batch_labels)

# === Build Hybrid Model (Image + Biomarkers + LBP) ===
def build_model():
    # Image input
    img_input = Input(shape=(224, 224, 3), name='image_input')
    base_model = ResNet152V2(include_top=False, input_tensor=img_input, weights='imagenet')
    x = GlobalAveragePooling2D()(base_model.output)

    # Biomarker input
    bio_input = Input(shape=(136,), name='biomarker_input')
    y = Dense(128, activation='relu')(bio_input)
    y = Dropout(0.3)(y)

    # LBP input
    lbp_input = Input(shape=(256,), name='lbp_input')
    z = Dense(128, activation='relu')(lbp_input)
    z = Dropout(0.3)(z)

    # Merge all
    combined = Concatenate()([x, y, z])
    final = Dense(128, activation='relu')(combined)
    final = Dropout(0.3)(final)
    output = Dense(1, activation='sigmoid')(final)

    model = Model(inputs=[img_input, bio_input, lbp_input], outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# === Prepare Data ===
X_train_bio, X_test_bio, y_train_bio, y_test_bio = load_biomarkers(train_csv, test_csv)

train_gen = HybridImageLBPGenerator(train_dir, X_train_bio, y_train_bio, batch_size, img_size)
test_gen = HybridImageLBPGenerator(test_dir, X_test_bio, y_test_bio, batch_size, img_size)

# === Train ===
model = build_model()
model.summary()

history = model.fit(train_gen, validation_data=test_gen, epochs=epochs)

# === Evaluate ===
def evaluate_model(model, test_gen, y_test):
    print("\nEvaluating Model...")
    y_pred_probs = []
    y_true = []

    for batch in tqdm(test_gen, total=len(test_gen), desc="Evaluating"):
        inputs, labels = batch
        preds = model.predict(inputs)
        y_pred_probs.extend(preds)
        y_true.extend(labels)

    y_pred = (np.array(y_pred_probs) > 0.5).astype(int).flatten()
    y_true = np.array(y_true).astype(int)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

evaluate_model(model, test_gen, y_test_bio)

# === Plotting ===
def plot_metrics(history):
    plt.figure(figsize=(12, 4))
    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('Model Accuracy')

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Model Loss')

    plt.tight_layout()
    plt.show()

plot_metrics(history)

